# Badge 4: Setup (run once, as ACCOUNTADMIN)

Creates the isolated environment the lesson notebooks run in:

| Course object | Isolated stand-in | Notes |
|---|---|---|
| ACCOUNTADMIN role | `BADGE4_DLKW_ADMIN` | holds the account privileges the course uses as ACCOUNTADMIN |
| SYSADMIN role | `BADGE4_DLKW_ROLE` | granted to `BADGE4_DLKW_ADMIN`, mirroring SYSADMIN under ACCOUNTADMIN |
| COMPUTE_WH | `BADGE4_DLKW_WH` | X-Small, auto-suspend, owned by `BADGE4_DLKW_ROLE` (Lesson 1 checklist) |
| UTIL_DB | `UTIL_DB` | shared across badges; owned by SYSADMIN (Lesson 1 checklist) |
| learner's Downloads folder | `@UTIL_DB.PUBLIC.BADGE4_DOWNLOADS` | holds the files the course has learners download and unzip |

Account-level steps (external volume, Marketplace listing, Iceberg database) run as the real ACCOUNTADMIN in the lesson notebooks and are labelled there. Object names (ZENAS_ATHLEISURE_DB, MELS_SMOOTHIE_CHALLENGE_DB, OPENSTREETMAP_DENVER, MY_ICEBERG_DB, ...) are the course's.

In [ ]:
# Isolation mapping (set once per notebook; SQL cells use these via {{...}})
ADMIN_ROLE    = "BADGE4_DLKW_ADMIN"   # stands in for the course's ACCOUNTADMIN
SYSADMIN_ROLE = "BADGE4_DLKW_ROLE"    # stands in for the course's SYSADMIN
WAREHOUSE     = "BADGE4_DLKW_WH"      # stands in for the course's COMPUTE_WH


def pdf(result):
    """A SQL cell's result as pandas (runtime 2.6+ returns a Snowpark DataFrame)."""
    return result.to_pandas() if hasattr(result, "to_pandas") else result

## Check for name collisions
Course database names are account-wide.

In [ ]:
%%sql -r existing_course_objects
use role accountadmin;
show databases like 'ZENAS_ATHLEISURE_DB';
show databases like 'MELS_SMOOTHIE_CHALLENGE_DB';
show databases like 'OPENSTREETMAP_DENVER';
show databases like 'MY_ICEBERG_DB';
show external volumes like 'ICEBERG_EXTERNAL_VOLUME';

## Roles and grants

In [ ]:
%%sql -r create_roles
use role accountadmin;
create role if not exists {{ADMIN_ROLE}} comment = 'Badge 4 stand-in for ACCOUNTADMIN';
create role if not exists {{SYSADMIN_ROLE}} comment = 'Badge 4 stand-in for SYSADMIN';
grant role {{SYSADMIN_ROLE}} to role {{ADMIN_ROLE}};
grant role {{ADMIN_ROLE}} to role sysadmin;
set badge_user = current_user();
grant role {{ADMIN_ROLE}} to user identifier($badge_user);
grant role {{SYSADMIN_ROLE}} to user identifier($badge_user);
-- SYSADMIN creates ZENAS_ATHLEISURE_DB and MELS_SMOOTHIE_CHALLENGE_DB (Lessons 2, 5)
grant create database on account to role {{SYSADMIN_ROLE}};
-- external tables auto-refresh and materialized views (Lesson 9)
grant create integration on account to role {{ADMIN_ROLE}};
-- ACCOUNTADMIN can read SNOWFLAKE.ACCOUNT_USAGE
grant imported privileges on database snowflake to role {{ADMIN_ROLE}};

## Warehouse (stands in for COMPUTE_WH; the Lesson 1 checklist says SYSADMIN owns it)

In [ ]:
%%sql -r create_warehouse
use role accountadmin;
create warehouse if not exists {{WAREHOUSE}}
  warehouse_size = xsmall auto_suspend = 60 auto_resume = true initially_suspended = true
  comment = 'Badge 4 stand-in for COMPUTE_WH';
grant ownership on warehouse {{WAREHOUSE}} to role {{SYSADMIN_ROLE}} copy current grants;

## UTIL_DB (shared across badges; home of the DORA GRADER)

In [ ]:
%%sql -r util_db_grants
use role accountadmin;
create database if not exists UTIL_DB comment = 'Shared utility database for badge course audits (DORA grader, course stages)';
grant ownership on database UTIL_DB to role sysadmin copy current grants;
grant ownership on schema UTIL_DB.PUBLIC to role sysadmin copy current grants;
grant usage on database UTIL_DB to role {{ADMIN_ROLE}};
grant usage on database UTIL_DB to role {{SYSADMIN_ROLE}};
grant usage, create stage, create workspace, create table on schema UTIL_DB.PUBLIC to role {{ADMIN_ROLE}};
grant usage, create table on schema UTIL_DB.PUBLIC to role {{SYSADMIN_ROLE}};

## Course data files (the learner's Downloads folder)
The course has learners download `sweatsuits.zip`, `metadata.zip`, `geoJSON_files.zip` and `cherry_creek_trail.parquet`, unzip them and upload the files to stages. The unzipped files are in `data/` beside these notebooks; upload them once:

```
snow sql -q "use role BADGE4_DLKW_ADMIN; put 'file://<path>/data/*' @UTIL_DB.PUBLIC.BADGE4_DOWNLOADS auto_compress=false overwrite=true"
```

In [ ]:
%%sql -r create_downloads_stage
use role {{ADMIN_ROLE}};
create stage if not exists UTIL_DB.PUBLIC.BADGE4_DOWNLOADS
  comment = 'Badge 4: stands in for the learner''s local Downloads folder'
  directory = (enable = true)
  encryption = (type = 'SNOWFLAKE_SSE');
grant read on stage UTIL_DB.PUBLIC.BADGE4_DOWNLOADS to role {{SYSADMIN_ROLE}};
alter stage UTIL_DB.PUBLIC.BADGE4_DOWNLOADS refresh;
list @UTIL_DB.PUBLIC.BADGE4_DOWNLOADS;

In [ ]:
expected = ['Bear_Creek_Trail.geojson', 'Big_Dry_Creek_Trail_to_Platte_River_to_Highline_Canal.geojson', 'Highline_Canal_Trail_North_Section.geojson', 'Highline_Canal_Trail_South_Section.geojson', 'burgundy_sweatsuit.png', 'charcoal_grey_sweatsuit.png', 'cherry_creek_trail.parquet', 'forest_green_sweatsuit.png', 'navy_blue_sweatsuit.png', 'orange_sweatsuit.png', 'pink_sweatsuit.png', 'product_coordination_suggestions.txt', 'purple_sweatsuit.png', 'red_sweatsuit.png', 'royal_blue_sweatsuit.png', 'sweatsuit_sizes.txt', 'swt_product_line.txt', 'yellow_sweatsuit.png']
found = sorted(n.split('/')[-1] for n in pdf(create_downloads_stage)['name'])
print('Missing course files:', sorted(set(expected) - set(found)) or 'none')

## Teardown (commented out)
Removes everything Badge 4 creates.

In [ ]:
%%sql -r teardown
-- use role accountadmin;
-- drop database if exists ZENAS_ATHLEISURE_DB;
-- drop database if exists MELS_SMOOTHIE_CHALLENGE_DB;
-- drop database if exists MY_ICEBERG_DB;
-- drop database if exists OPENSTREETMAP_DENVER;
-- drop external volume if exists ICEBERG_EXTERNAL_VOLUME;
-- drop table if exists UTIL_DB.PUBLIC.MY_DATA_TYPES;
-- drop stage if exists UTIL_DB.PUBLIC.BADGE4_DOWNLOADS;
-- drop warehouse if exists {{WAREHOUSE}};
-- drop role if exists {{SYSADMIN_ROLE}};
-- drop role if exists {{ADMIN_ROLE}};
select 'teardown is commented out' as status;